# Giving Agents Access to Tools

Tool calling turns an LLM from a text generator into an agent that can act on the world. This notebook covers both the *protocol* — how function calling works at the wire level in the OpenAI API — and the *design* — how to build composable, validated tool abstractions. We start with raw API function calling, examine an XML-based alternative, discuss schema design principles, then walk through the CDA library's `Tool` ABC and `ToolRegistry`.

## The Chat Completions API

The simplest possible LLM interaction is a single chat completion: send a list of messages, receive a reply. We use `AsyncOpenAI` pointing at OpenRouter — an API gateway that routes to hundreds of models under a single API key. The client reads `OPENROUTER_API_KEY` and `OPENROUTER_BASE_URL` from the environment.

**Setup.** Loading dependencies and initializing the client:

In [ ]:
import os
import json
import asyncio
from dotenv import load_dotenv
from openai import AsyncOpenAI

load_dotenv()

client = AsyncOpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url=os.environ.get("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"),
)

MODEL = "anthropic/claude-sonnet-4"

The `chat.completions.create` method accepts a list of messages with roles `system`, `user`, and `assistant`. We pass `stream=False` for a single blocking response.

Non-streaming call:

In [ ]:
async def chat(messages, model=MODEL, **kwargs):
    response = await client.chat.completions.create(
        model=model,
        messages=messages,
        **kwargs,
    )
    return response.choices[0].message.content

reply = await chat([{"role": "user", "content": "What is 2 + 2?"}])
print(reply)

For streaming, we set `stream=True` and iterate over chunks. Each chunk carries a `delta` — the incremental token being added to the response.

Streaming call:

In [ ]:
async def stream_chat(messages, model=MODEL, **kwargs):
    stream = await client.chat.completions.create(
        model=model,
        messages=messages,
        stream=True,
        **kwargs,
    )
    async for chunk in stream:
        delta = chunk.choices[0].delta.content or ""
        print(delta, end="", flush=True)
    print()

await stream_chat([{"role": "user", "content": "Name three planets in one sentence."}])

:::{.callout-note}
The `stream=True` variant lets a UI display tokens as they arrive, giving the impression of faster responses. The total latency (time-to-completion) is the same; only the time-to-first-token changes.

:::

## Native Function Calling

The OpenAI function-calling protocol lets the model signal that it wants to call an external function instead of (or before) producing a final answer. The model never actually executes anything — that happens in our code. The protocol has four elements: (1) a `tools` list sent in the request, (2) a `tool_calls` array in the assistant response, (3) execution on our side, (4) a `role: "tool"` message feeding the result back.

### The Protocol

We define a weather tool as a JSON Schema dict and a corresponding Python function. The schema tells the model the tool's name, description, and parameter types.

**Schema.** Defining the weather function and its JSON Schema descriptor:

In [ ]:
import requests

def get_weather(latitude: float, longitude: float) -> dict:
    """Get current weather for coordinates (temperature in Celsius, wind speed in km/h)."""
    resp = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": latitude,
            "longitude": longitude,
            "current": "temperature_2m,wind_speed_10m,relative_humidity_2m,precipitation",
        },
        timeout=10,
    )
    return resp.json()["current"]

WEATHER_SCHEMA = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": (
            "Get current weather at a location. Use this when the user asks about "
            "current weather, temperature, or wind conditions at a specific place."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "latitude":  {"type": "number", "description": "Latitude in decimal degrees"},
                "longitude": {"type": "number", "description": "Longitude in decimal degrees"},
            },
            "required": ["latitude", "longitude"],
        },
    },
}

We now send a user message along with the tool definition. The model will return an assistant message whose `tool_calls` field lists the function(s) it wants to invoke.

First call — model produces a `tool_calls` response, not a text answer:

In [ ]:
messages = [{"role": "user", "content": "What's the weather like in Manila right now?"}]

# (1) First call: model produces a tool_call, not a text answer
response = await client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=[WEATHER_SCHEMA],
)
assistant_msg = response.choices[0].message
print("finish_reason:", response.choices[0].finish_reason)
print("tool_calls:", assistant_msg.tool_calls)

The model responded with `finish_reason="tool_calls"` and a `tool_calls` list. Each call has an `id`, `function.name`, and `function.arguments` (a JSON string). We execute the function and return the result as a `role: "tool"` message, then call the API again.

Executing the tool calls and completing the round-trip:

In [ ]:
TOOL_MAP = {"get_weather": get_weather}

def execute_tool_calls(tool_calls):
    results = []
    for tc in tool_calls:
        name = tc.function.name                     # (1) tool name
        args = json.loads(tc.function.arguments)    # (2) parse JSON args
        result = TOOL_MAP[name](**args)             # (3) execute locally
        results.append({
            "role": "tool",
            "tool_call_id": tc.id,
            "content": json.dumps(result),
        })
    return results

# Add the assistant message (with tool_calls) to history
messages.append(assistant_msg)

# Execute and add tool results
tool_results = execute_tool_calls(assistant_msg.tool_calls)
messages.extend(tool_results)

# (4) Second call: model uses results to produce the final answer
final_response = await client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=[WEATHER_SCHEMA],
)
print(final_response.choices[0].message.content)

1. `tc.function.name` identifies which tool to dispatch.
2. `tc.function.arguments` is a JSON string — we must parse it before passing to the function.
3. The function executes locally with the parsed keyword arguments.
4. The second API call completes the turn: the model reads the tool result and produces a natural-language answer.

:::{.callout-note}
The "execute and loop" pattern — send → `tool_calls` → execute → send results → repeat — is the embryonic form of the agent loop we build in depth in the next notebook.

:::

### Multi-Tool Calling

The model can request multiple tool calls in a single response. We register a calculator alongside the weather tool and send a compound query.

**Schema.** Defining the calculator function and registering both tools:

In [ ]:
def calculate(expression: str) -> dict:
    """Evaluate a mathematical expression safely."""
    try:
        result = eval(expression, {"__builtins__": {}}, {})  # noqa: S307
        return {"result": result}
    except Exception as e:
        return {"error": str(e)}

CALC_SCHEMA = {
    "type": "function",
    "function": {
        "name": "calculate",
        "description": "Evaluate a mathematical expression. Use this for arithmetic.",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {"type": "string", "description": "Python arithmetic expression, e.g. '37 * 43'"},
            },
            "required": ["expression"],
        },
    },
}

TOOL_MAP["calculate"] = calculate
ALL_TOOLS = [WEATHER_SCHEMA, CALC_SCHEMA]

We send a single message that requires both tools. Some models return two `tool_calls` in one response (parallel tool calling); others return one at a time.

Parallel tool call request:

In [ ]:
messages2 = [{
    "role": "user",
    "content": "What is 37 times 43? Also, what's the weather in Tokyo?"
}]

resp2 = await client.chat.completions.create(
    model=MODEL, messages=messages2, tools=ALL_TOOLS,
)
print(f"Number of tool calls: {len(resp2.choices[0].message.tool_calls or [])}")
for tc in resp2.choices[0].message.tool_calls or []:
    print(f"  {tc.function.name}({tc.function.arguments})")

## XML-Based Tool Dispatch

Native function calling requires API support. Models accessed via plain completion endpoints (or fine-tuned models without function-calling training) can still use tools through a text-based protocol: we describe tools in the system prompt and instruct the model to emit XML tags for tool calls.

The system prompt describes available tools and the expected call format. We use `<tool_call>` tags with `<name>` and `<params>` children.

**Protocol.** Defining the system prompt and a parser for XML tool calls:

In [ ]:
import re

XML_SYSTEM_PROMPT = """You have access to the following tools:

<tools>
<tool name="get_weather">
  Get current weather at a location given latitude and longitude.
  Params: latitude (float), longitude (float)
</tool>
<tool name="calculate">
  Evaluate a mathematical expression.
  Params: expression (string, e.g. "37 * 43")
</tool>
</tools>

When you need to call a tool, emit:
<tool_call>
<name>TOOL_NAME</name>
<params>{"key": value, ...}</params>
</tool_call>

After seeing a <tool_result>, continue your response normally.
Do NOT call a tool if the answer is already known."""

def parse_tool_calls(text: str) -> list[dict]:
    """Extract tool calls from XML-formatted model output."""
    calls = []
    pattern = r"<tool_call>\s*<name>(.*?)</name>\s*<params>(.*?)</params>\s*</tool_call>"
    for name, params_str in re.findall(pattern, text, re.DOTALL):
        calls.append({"name": name.strip(), "params": json.loads(params_str.strip())})
    return calls

We run a completion loop: generate → check for `<tool_call>` → execute → inject result → continue.

XML dispatch loop:

In [ ]:
async def xml_agent(user_message: str) -> str:
    """Run one turn of XML-based tool dispatch."""
    messages = [
        {"role": "system", "content": XML_SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]

    while True:
        resp = await client.chat.completions.create(
            model=MODEL,
            messages=messages,
            stop=["</tool_call>"],   # (1) stop after each call for clean parsing
        )
        text = resp.choices[0].message.content or ""
        calls = parse_tool_calls(text + "</tool_call>")  # (2) re-close the tag

        if not calls:
            return text  # (3) no more tool calls → done

        # Execute and inject results
        result_parts = []
        for call in calls:
            result = TOOL_MAP[call["name"]](**call["params"])
            result_parts.append(f"<tool_result>{json.dumps(result)}</tool_result>")

        messages.append({"role": "assistant", "content": text + "</tool_call>"})
        messages.append({"role": "user", "content": "\n".join(result_parts)})

answer = await xml_agent("What is 37 times 43?")
print(answer)

1. Stopping generation at `</tool_call>` ensures we can parse one call at a time before continuing.
2. We re-attach the closing tag before parsing, since the stop token was consumed.
3. When no `<tool_call>` tags appear in the output, the model is done; we return the final text.

:::{.callout-note}
XML dispatch is model-agnostic but more fragile — the model might not follow the format consistently. Native function calling is preferred when the API supports it. XML remains useful for models accessed via non-OpenAI endpoints or for custom protocols.

:::

## Designing Tool Schemas

A tool schema is the model's only specification of what a tool does and when to use it. A vague schema leads to misuse; a precise schema enables reliable tool selection. Anthropic calls this the **Agent-Computer Interface** (ACI) design problem — the interface between the agent and its computational environment.

### Schema Design Principles

Four principles apply to every tool schema:

1. **Name with a verb-noun pair.** `search_files` not `files`; `get_weather` not `weather`. The name is the model's first signal about what the tool does.
2. **Describe _when_ to use it, not just _what_ it does.** "Use this when the user asks about current weather" guides the model's decision; "Returns weather data" does not.
3. **Keep parameters atomic and constrained.** Prefer `enum` over free text where the options are bounded. Avoid compound parameters that encode multiple pieces of information.
4. **Make errors actionable.** Return `"File not found: foo.py. Available files: bar.py, baz.py"` not just `"File not found"`.

We compare a bad schema with a good one for the same conceptual tool.

Bad vs. good schema side by side:

In [ ]:
# BAD: vague name, unhelpful description, no constraints
BAD_SCHEMA = {
    "type": "function",
    "function": {
        "name": "files",
        "description": "Do something with files.",
        "parameters": {
            "type": "object",
            "properties": {
                "action": {"type": "string"},
                "data": {"type": "string"},
            },
        },
    },
}

# GOOD: verb-noun name, when-to-use description, constrained parameters
GOOD_SCHEMA = {
    "type": "function",
    "function": {
        "name": "read_file",
        "description": (
            "Read the contents of a file. Use this when you need to examine source "
            "code, configuration files, or any text file. Returns the file contents "
            "as a string. Error if the file does not exist."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "path": {
                    "type": "string",
                    "description": "Absolute or relative path to the file",
                },
            },
            "required": ["path"],
        },
    },
}

## The CDA Tool ABC

The CDA library (at `src/notebooks/agent/`) provides a `Tool` abstract base class that enforces the schema/execute contract, handles Pydantic validation, and integrates with the approval system. Every builtin tool and custom tool in the library subclasses `Tool`.

### The Tool Base Class

The key design decisions in `tools/base.py` are: (1) the schema is a Pydantic `BaseModel` subclass (or a raw dict), enabling automatic validation; (2) `ToolKind` categorizes tools for approval gating; (3) `ToolResult` is a structured return type, not a plain string.

Importing the tool primitives:

In [ ]:
from notebooks.agent.tools.base import Tool, ToolKind, ToolResult, ToolInvocation
from notebooks.agent.config import Config
from pydantic import BaseModel

`ToolKind` has five values that map directly to approval policy logic: `READ`, `WRITE`, `SHELL`, `NETWORK`, `MEMORY`. READ tools are always auto-approved; the others are gated depending on the `ApprovalPolicy` in use.

Inspecting the `ToolKind` enum:

In [ ]:
for kind in ToolKind:
    print(f"  ToolKind.{kind.name} = {kind.value!r}")

### Building a Custom Tool

We implement a `DiceRollTool` as a concrete example. The pattern is: define a Pydantic schema for parameters, implement `execute()`, return a `ToolResult`.

Custom tool implementation:

In [ ]:
import random

class DiceRollParams(BaseModel):
    sides: int = 6
    count: int = 1

class DiceRollTool(Tool):
    name = "roll_dice"
    description = (
        "Roll one or more dice and return the results. "
        "Use this when the user asks for a random number or dice roll."
    )
    kind = ToolKind.READ
    schema = DiceRollParams

    async def execute(self, invocation: ToolInvocation) -> ToolResult:
        params = DiceRollParams(**invocation.params)              # (1) validated by Pydantic
        rolls = [random.randint(1, params.sides) for _ in range(params.count)]
        total = sum(rolls)
        output = f"Rolled {params.count}d{params.sides}: {rolls} (total: {total})"
        return ToolResult.success_result(output)                  # (2) structured return

1. Pydantic validates and coerces the raw `params` dict on construction. Invalid input raises `ValidationError` before any random numbers are generated.
2. `ToolResult.success_result` wraps the output string in a structured dataclass with `success=True` and `error=None`.

We test validation directly. Invalid parameters produce a structured error before `execute()` is ever called.

Validation — valid and invalid params:

In [ ]:
config = Config()
tool = DiceRollTool(config)

# Valid params
errors = tool.validate_params({"sides": 20, "count": 3})
print("Valid params errors:", errors)  # []

# Invalid params (sides must be int)
errors = tool.validate_params({"sides": "many"})
print("Invalid params errors:", errors)

We also inspect the OpenAI-format schema generated from the Pydantic model.

Schema generation from the Pydantic model:

In [ ]:
import json
schema = tool.to_openai_schema()
print(json.dumps(schema, indent=2))

### The Registry Pattern

The `ToolRegistry` is the central dispatcher. It stores tools by name, filters by `allowed_tools`, and provides `get_schemas()` in OpenAI format. We register our custom tool and invoke it programmatically.

Registering and invoking via the registry:

In [ ]:
from notebooks.agent.tools.registry import ToolRegistry

registry = ToolRegistry(config)
registry.register(DiceRollTool(config))

# Invoke by name
result = await registry.invoke(
    name="roll_dice",
    params={"sides": 6, "count": 2},
    cwd=config.cwd,
)
print("Success:", result.success)
print("Output:", result.output)

The `allowed_tools` config field restricts which registered tools are visible. This is used to give different agents different capability sets.

Restricting the default registry to a named subset:

In [ ]:
from notebooks.agent.tools.registry import create_default_registry

# Restrict to only two tools
restricted_config = Config(allowed_tools=["roll_dice", "read_file"])
full_registry = create_default_registry(restricted_config)
full_registry.register(DiceRollTool(restricted_config))

schemas = full_registry.get_schemas()
print("Available tools:", [s["name"] for s in schemas])

:::{.callout-caution}
`get_schemas()` only returns tools that pass the `allowed_tools` filter, but `invoke()` dispatches against the full internal `_tools` dict. A tool registered after the allowed list was set will be invocable by name even if it does not appear in `get_schemas()`. Filter consistently at both registration and invocation boundaries if isolation is required.

:::

## Appendix: Survey of Builtin Tools

The CDA library ships 11 builtin tools loaded by `create_default_registry()`. The table below serves as a quick reference for later notebooks.

| Tool | Kind | Description |
|------|------|-------------|
| `read_file` | READ | Read a file's contents |
| `write_file` | WRITE | Create or overwrite a file |
| `edit_file` | WRITE | Targeted string replacement in a file |
| `shell` | SHELL | Run a shell command in the working directory |
| `list_dir` | READ | List files and subdirectories |
| `grep` | READ | Search file contents with a regex pattern |
| `glob` | READ | Find files matching a glob pattern |
| `fetch_url` | NETWORK | Fetch content from a URL |
| `web_search` | NETWORK | Search the web (requires search API key) |
| `memory` | MEMORY | Persistent key-value store (`~/.cda/memory.json`) |
| `todo` | WRITE | Manage a task list for the current session |

: Builtin tools in the CDA library. {tbl-colwidths="[20,15,65]"}

---

■